In [2]:
"""
Py-Microgrid Multiple Locations Hybrid System Simulation Example
-----------------------------------
This example demonstrates how to:
1. Set up a hybrid system simulation for multiple locations
2. Download solar and wind resource data
3. Configure system parameters
4. Run optimization for each location
5. Analyze and save results

Required files:
- Base YAML configuration file
- CSV file containing location data
"""

import os
import pandas as pd
from typing import Dict, List, Any

# Set NREL API key FIRST before any other imports to avoid timing issues
from py_microgrid.utilities.keys import set_developer_nrel_gov_key
set_developer_nrel_gov_key('ZaurwKOnwDUp8rMyNBIxI4XiBo3b7L5oruTi0VX3')

# Required imports (after API key is set)
from py_microgrid.utilities import ConfigManager
from py_microgrid.tools.optimization.system_optimizer import SystemOptimizer  
from py_microgrid.tools.optimization import LoadAnalyzer
from py_microgrid.tools.optimization import EconomicCalculator
from py_microgrid.simulation.resource_files import ResourceDataManager

def optimize_single_location(latitude: float, longitude: float, location_id: str = ""):
    """Optimize a single location using EXACT same code as quick_start_example.ipynb"""
    print(f"\n--- Processing location {location_id} at ({latitude}, {longitude}) ---")
    
    # Initialize resource manager for downloading data
    resource_manager = ResourceDataManager(
        api_key='ZaurwKOnwDUp8rMyNBIxI4XiBo3b7L5oruTi0VX3',
        email='hanrong.h99@gmail.com'
    )

    # Download resource data
    solar_path = resource_manager.download_solar_data(
        latitude=latitude,
        longitude=longitude,
        year="2022" 
    )
    wind_path = resource_manager.download_wind_data(
        latitude=latitude,
        longitude=longitude,
        start_date="20220101",  
        end_date="20221231"
    )

    # Load and update YAML configuration
    yaml_file_path = "../input_yaml/multiple_locations_config.yaml"
    config_manager = ConfigManager()
    config = config_manager.load_yaml_safely(yaml_file_path)

    # Update configuration with location and resource files
    config['site']['data']['lat'] = latitude
    config['site']['data']['lon'] = longitude
    config['site']['solar_resource_file'] = solar_path.replace('\\', '/')  # Normalize path separators
    config['site']['wind_resource_file'] = wind_path.replace('\\', '/')

    # Save updated configuration
    config_manager.save_yaml_safely(config, yaml_file_path)

    # Initialize components for optimization with hardcoded economic parameters
    economic_calculator = EconomicCalculator(
        discount_rate=0.0588,    # 5.88% discount rate
        project_lifetime=25      # 25 year project lifetime
    )

    optimizer = SystemOptimizer(
        yaml_file_path=yaml_file_path,
        economic_calculator=economic_calculator,
        enable_flexible_load=True,
        max_load_reduction_percentage=0.2   # max load reducntion
    )

    # Define optimization bounds
    config = config_manager.load_yaml_safely(yaml_file_path)
    grid_enabled = config.get('technologies', {}).get('grid', {}).get('enabled', True)  # Set to True for grid-connected

    if grid_enabled:
        # 6-component optimization: PV, Wind, Battery kWh, Battery kW, Genset, Grid
        bounds = [
            (5000, 50000),    # PV capacity (kW)
            (5, 50),          # Wind turbines
            (5000, 30000),    # Battery capacity (kWh)
            (1000, 10000),    # Battery power (kW)
            (5000, 20000),    # Genset capacity (kW)
            (5000, 20000)     # Grid capacity (kW)
        ]
    else:
        # 5-component optimization: PV, Wind, Battery kWh, Battery kW, Genset
        bounds = [
            (5000, 50000),    # PV capacity (kW)
            (5, 50),          # Wind turbines
            (5000, 30000),    # Battery capacity (kWh)
            (1000, 10000),    # Battery power (kW)
            (5000, 20000)     # Genset capacity (kW)
        ]

    # Define initial conditions (start at 10% of the range for each variable)
    initial_conditions = [
        [bound[0] + (bound[1] - bound[0]) * 0.1 for bound in bounds]
    ]

    print(f"Running optimization with {'6' if grid_enabled else '5'} components")
    print(f"Grid enabled: {grid_enabled}")

    # Run the optimization
    result = optimizer.optimize_system(bounds, initial_conditions)

    # Print final results
    if result:
        print("\nOptimization Results:")
        print(f"PV Capacity: {result['PV Capacity (kW)']:.2f} kW")
        print(f"Wind Capacity: {result['Wind Turbine Capacity (kW)']:.2f} kW")
        print(f"Battery Capacity: {result['Battery Energy Capacity (kWh)']:.2f} kWh")
        print(f"Battery Power: {result['Battery Power Capacity (kW)']:.2f} kW")
        print(f"Genset Capacity: {result['Genset Capacity (kW)']:.2f} kW")
        if grid_enabled:
            print(f"Grid Capacity: {result['Grid Capacity (kW)']:.2f} kW")
        print(f"\nLCOE: ${result['System LCOE ($/kWh)']:.4f}/kWh")
        print(f"Net Present Cost: ${result['Net Present Cost ($)']:.2f}")
        print(f"CO2 Emissions: {result['Total CO2 emissions (tonne)']:.2f} tonnes")
        print(f"Demand Met: {result['Demand Met Percentage']:.2f}%")
        
        return {'Latitude': latitude, 'Longitude': longitude, 'Location ID': location_id, **result}
    else:
        print("Optimization failed to find a solution")
        return {}

def main():
    """Main execution - single location example."""
    test_location = {'latitude': -33.5265, 'longitude': 149.1588, 'id': "TEST_LOCATION"}
    
    result = optimize_single_location(
        latitude=test_location['latitude'],
        longitude=test_location['longitude'],
        location_id=test_location['id']
    )
    
    if result:
        results_df = pd.DataFrame([result])
        os.makedirs("simulation_results", exist_ok=True)
        csv_filename = os.path.join(
            "../simulation_results",
            f"simulation_results_{test_location['latitude']}_{test_location['longitude']}.csv"
        )
        results_df.to_csv(csv_filename, index=False)
        print(f"\n✓ Results saved to: {csv_filename}")
        print("✓ Single location optimization completed successfully!")
    else:
        print("\n✗ Single location optimization failed")

def process_multiple_locations():
    """Process multiple locations from CSV file."""
    # Configuration
    csv_path = "../deposit_data/auCopper_chunk_0.csv"  # Path to your locations CSV
    output_path = "../simulation_results/simulation_results_all.csv"
    
    # Load location data
    try:
        location_data = pd.read_csv(csv_path)
        print(f"Loaded {len(location_data)} locations from {csv_path}")
    except FileNotFoundError:
        print(f"Error: Could not find location data file: {csv_path}")
        return
    
    # Process each location
    results = []
    for idx, row in location_data.iterrows():
        result = optimize_single_location(
            latitude=row['Latitude'], 
            longitude=row['Longitude'], 
            location_id=row.get('Location_ID', f"LOC_{idx}")
        )
        if result:
            results.append(result)
        else:
            print(f"--- Failed to process location {row.get('Location_ID', f'LOC_{idx}')} ---")
    
    # Save all results
    if results:
        results_df = pd.DataFrame(results)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        results_df.to_csv(output_path, index=False)
        print(f"\n✓ All results saved to: {output_path}")
        print(f"✓ Successfully processed {len(results)}/{len(location_data)} locations")
        
        # Display summary statistics
        print("\n--- Summary Statistics ---")
        print(f"  Average LCOE: ${results_df['System LCOE ($/kWh)'].mean():.4f}/kWh")
        print(f"  LCOE Range: ${results_df['System LCOE ($/kWh)'].min():.4f} - ${results_df['System LCOE ($/kWh)'].max():.4f}/kWh")
        print(f"  Average Demand Met: {results_df['Demand Met Percentage'].mean():.1f}%")
    else:
        print("\n✗ No successful optimizations completed")


if __name__ == "__main__":
    # To run a single location optimization:
    main()
    
    # To run multiple locations, comment out main() and uncomment the line below:
    # process_multiple_locations()


--- Processing location TEST_LOCATION at (-33.5265, 149.1588) ---
Using existing solar data file: /mnt/c/Users/Hanrong Huang/OneDrive - UNSW/Desktop/py_microgrid/py_microgrid/simulation/resource_files/solar/-33.5265_149.1588_psmv3_60_2022.csv
Using existing wind data file: /mnt/c/Users/Hanrong Huang/OneDrive - UNSW/Desktop/py_microgrid/py_microgrid/simulation/resource_files/wind/-33.5265_149.1588_NASA_2022_60min_50m.srw
✓ Successfully loaded cost configs from: /mnt/c/Users/Hanrong Huang/OneDrive - UNSW/Desktop/py_microgrid/py_microgrid/simulation/config
Running optimization with 6 components
Grid enabled: True

Calculating costs for: PV:9500kW, Wind:6000.0kW, Bat:7500kWh, Gen:6500kW, Grid:6500kW
  PV Cost: $22,562,500
  Wind Cost: $21,000,000
  Battery Cost: $4,975,000
  Genset Cost: $460,519,417 (Fuel: $354,054,870, Replace: $54,925,000, Carbon: $7,724,297)
  Grid Cost: $325,000

Calculating costs for: PV:9975kW, Wind:6000.0kW, Bat:7500kWh, Gen:6500kW, Grid:6500kW
  PV Cost: $23,690,